In [10]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [11]:
class CrashState(TypedDict) :
    input : str
    step1 : str
    step2 : str
    step3 : str

In [12]:
def step_1(state : CrashState) -> CrashState :
     print('Step 1 Executed!')
     return {'step1' : 'done', "input" : state['input']}

def step_2(state : CrashState) -> CrashState :
     print('Step 2 hanging...  now manually interupt from the code (Stop Button)')
     time.sleep(30)
     return {'step2' : "Done"}

def step_3(state : CrashState) -> CrashState :
     print('Step 3 Executed!')
     return {"done" : True}

In [13]:
graph = StateGraph(CrashState)

graph.add_node('Step_1', step_1)
graph.add_node('Step_2', step_2)
graph.add_node('Step_3', step_3)

graph.add_edge(START, 'Step_1')
graph.add_edge('Step_1', 'Step_2')
graph.add_edge('Step_2', 'Step_3')
graph.add_edge('Step_3', END)

check = InMemorySaver()

workflow = graph.compile(checkpointer=check)

In [14]:
try:
    print("Graph is running please interupt manually!")
    workflow.invoke({"input" : "start"}, config={"configurable": {"thread_id" : '1'}})
except KeyboardInterrupt:
    print("manually interupted !")

Graph is running please interupt manually!
Step 1 Executed!
Step 2 hanging...  now manually interupt from the code (Stop Button)
Step 3 Executed!


In [15]:
workflow.get_state({"configurable": {"thread_id" : '1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'Done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18786b-6931-6d72-8003-f9537f2eb9bb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-24T17:40:08.365809+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18786b-692d-6f8a-8002-603d43f54379'}}, tasks=(), interrupts=())

In [ ]:
list(workflow.get_state_history({"configurable": {"thread_id" : '1'}}))

<generator object Pregel.get_state_history at 0x0000015D7EFFBE20>

In [17]:
final_state = workflow.invoke(None, config={'configurable': {'thread_id' : '1'}})
print("\n Final State: ", final_state)


 Final State:  {'input': 'start', 'step1': 'done', 'step2': 'Done'}


In [18]:
workflow.get_state({"configurable": {"thread_id" : '1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'Done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18786b-6931-6d72-8003-f9537f2eb9bb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-24T17:40:08.365809+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18786b-692d-6f8a-8002-603d43f54379'}}, tasks=(), interrupts=())

In [19]:
workflow.get_state_history({"configurable": {"thread_id" : '1'}})

<generator object Pregel.get_state_history at 0x0000015D7F198040>